In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from helper import *
import numpy as np
# ignore warning
warnings.filterwarnings("ignore")

# make sure printed values are not truncated
pd.set_option('display.max_rows', None)



In [3]:
accounts_df = load_accounts()
events_df = load_events()
support_tickets_df = load_support_tickets()
feature_usage_df = load_feature_usage()
subscription_df = load_subscriptions()

## Account Table

In [4]:
accounts_df.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [5]:
accounts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   account_id       500 non-null    object
 1   account_name     500 non-null    object
 2   industry         500 non-null    object
 3   country          500 non-null    object
 4   signup_date      500 non-null    object
 5   referral_source  500 non-null    object
 6   plan_tier        500 non-null    object
 7   seats            500 non-null    int64 
 8   is_trial         500 non-null    bool  
 9   churn_flag       500 non-null    bool  
dtypes: bool(2), int64(1), object(7)
memory usage: 32.4+ KB


In [6]:
accounts_df.describe()

,seats
count,500.000000
mean,20.560000
std,21.044718
min,1.000000
25%,5.000000
50%,15.000000
75%,28.000000
max,163.000000


In [13]:
# convert the date column to the appropriate type
accounts_df["signup_date"] = pd.to_datetime(accounts_df["signup_date"])

# check for duplicates
duplicate_total = accounts_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# check for account_id uniqueness
unique_id = (accounts_df.account_id.nunique() / len(accounts_df)) * 100
print(f"The account ids are {int(unique_id)} % unique")

# check for the earliest and latest date in the table
min_date = accounts_df["signup_date"].min()
max_date = accounts_df["signup_date"].max()
print(f"The earliest date in the account table is: {min_date}\nThe latest date in the account table is : {max_date}")


There are 0 duplicate(s) in the table
The account ids are 100 % unique
The earliest date in the account table is: 2023-01-02 00:00:00
The latest date in the account table is : 2024-12-31 00:00:00


In [25]:

# check for typos and related issues in the account table columns
concerned_cols = ["industry", "country", "referral_source", "plan_tier"]
for col in concerned_cols:
    result = accounts_df[col].value_counts()
    print('--' * 20)
    print(result.to_string())
    print('--' * 20)

----------------------------------------
industry
DevTools         113
FinTech          112
Cybersecurity    100
HealthTech        96
EdTech            79
----------------------------------------
----------------------------------------
country
US    291
UK     58
IN     49
AU     32
DE     25
CA     23
FR     22
----------------------------------------
----------------------------------------
referral_source
organic    114
other      103
ads         98
event       96
partner     89
----------------------------------------
----------------------------------------
plan_tier
Pro           178
Basic         168
Enterprise    154
----------------------------------------


## Events Table

In [26]:
events_df.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,None
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [27]:
events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    object 
 1   account_id                600 non-null    object 
 2   churn_date                600 non-null    object 
 3   reason_code               600 non-null    object 
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    object 
dtypes: bool(3), float64(1), object(5)
memory usage: 30.0+ KB


In [28]:
events_df.describe()

,refund_amount_usd
count,600.000000
mean,14.420417
std,39.224591
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,392.920000


- Notice the difference between the mean 39.22 and the median 0.00 ==> skewed data !!

In [45]:
# check for missing values
miss_total = events_df.isna().sum().sum()
print(f"The table contains {miss_total} missing value(s).")

# check for duplicates
duplicate_total = events_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# chech for uniqueness in the churn_event_id column
unique_id = (events_df.churn_event_id.nunique() / len(events_df)) * 100
print(f"The churn_event_id are {int(unique_id)} % unique")


# convert the date column
events_df["churn_date"] = pd.to_datetime(events_df["churn_date"])
min_date = events_df["churn_date"].min()
max_date = events_df["churn_date"].max()
print(f"The earliest date in the account table is: {min_date}\nThe latest date in the account table is : {max_date}")


The table contains 148 missing value(s).
There are 0 duplicate(s) in the table
The churn_event_id are 100 % unique
The earliest date in the account table is: 2023-01-25 00:00:00
The latest date in the account table is : 2024-12-31 00:00:00


- The `feedback_text` column contains 148 missing value.In the description of the column it is said that comments are optional so we can treat those missing values as 'no comment'.

- We can use this code: 

```python
    events_df["feedback_text"] = events_df["feedback_text"].fillna("no comment")
``` 

In [47]:
# check for typos and other issues in the object type columns
events_df["reason_code"].value_counts()

reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64

## Support Tickets Table

In [48]:
support_tickets_df.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False


In [49]:
support_tickets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ticket_id                    2000 non-null   object        
 1   account_id                   2000 non-null   object        
 2   submitted_at                 2000 non-null   datetime64[ns]
 3   closed_at                    2000 non-null   datetime64[ns]
 4   resolution_time_hours        2000 non-null   float64       
 5   priority                     2000 non-null   object        
 6   first_response_time_minutes  2000 non-null   int64         
 7   satisfaction_score           1175 non-null   float64       
 8   escalation_flag              2000 non-null   bool          
dtypes: bool(1), datetime64[ns](2), float64(2), int64(1), object(3)
memory usage: 127.1+ KB


In [50]:
support_tickets_df.describe()

,submitted_at,closed_at,resolution_time_hours,first_response_time_minutes,satisfaction_score
count,2000,2000,2000.000000,2000.000000,1175.000000
mean,2024-01-05 14:08:09.599999744,2024-01-07 01:59:49.200000,35.861000,88.480000,3.981277
min,2023-01-02 00:00:00,2023-01-03 03:00:00,1.000000,1.000000,3.000000
25%,2023-07-09 18:00:00,2023-07-10 21:00:00,17.000000,43.000000,3.000000
50%,2024-01-06 12:00:00,2024-01-08 05:00:00,35.000000,88.000000,4.000000
75%,2024-07-09 06:00:00,2024-07-10 11:00:00,54.000000,131.000000,5.000000
max,2024-12-31 00:00:00,2024-12-31 19:00:00,72.000000,180.000000,5.000000
std,NaN,NaN,21.138427,51.531877,0.809646


In [51]:
# check for missing values
miss_total = support_tickets_df.isna().sum().sum()
print(f"The table contains {miss_total} missing value(s).")

# check for duplicates
duplicate_total = support_tickets_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# chech for uniqueness in the churn_event_id column
unique_id = (support_tickets_df.ticket_id.nunique() / len(support_tickets_df)) * 100
print(f"The churn_event_id are {int(unique_id)} % unique")


# convert the date column
sub_min_date = support_tickets_df["submitted_at"].min()
sub_max_date = support_tickets_df["submitted_at"].max()

clos_min_date = support_tickets_df["closed_at"].min()
clos_max_date = support_tickets_df["closed_at"].max()

print(f"The earliest submission date in the support ticket table is: {sub_min_date} and the latest is : {sub_max_date}")
print(f"The earliest closed date in the support ticket table is: {clos_min_date} and the latest is : {clos_max_date}")


The table contains 825 missing value(s).
There are 0 duplicate(s) in the table
The churn_event_id are 100 % unique
The earliest submission date in the support ticket table is: 2023-01-02 00:00:00 and the latest is : 2024-12-31 00:00:00
The earliest closed date in the support ticket table is: 2023-01-03 03:00:00 and the latest is : 2024-12-31 19:00:00
